# Part 1
The Refactored Loop

In [ ]:
#-----part1_refactored_loop.py--------#
#----------Andrew_Martin--------------#

import numpy as np
from week5_part2 import relu
from week5_part3 import relu2deriv, one_step


tells = np.array([[1, 0, 1],   # foot shift, no guard drop, exhale
                   [0, 1, 1],   # no shift, guard drop, exhale
                   [0, 0, 1],   # only exhale
                   [1, 1, 1]])  # all three (the bluff)

strike = np.array([[1, 1, 0, 0]]).T  # column vector, shape (4, 1)


def train(tells, strike, alpha, epochs, hidden_size, seed):
    """
    Same Week 5 algorithm (single hidden layer, ReLU on the hidden layer,
    no output activation, stochastic gradient descent, one update per
    sensing), rewritten so each pass through the network is organized into
    the four movements from lecture: FORWARD, COMPARE, BACKWARD, UPDATE.

    Returns: (weights_0_1, weights_1_2, error_history)
    """
    np.random.seed(seed)
    n_features = tells.shape[1]

    weights_0_1 = 2 * np.random.random((n_features, hidden_size)) - 1  # (n_features, hidden_size)
    weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1           # (hidden_size, 1)

    error_history = []

    # Tuesday's lecture loop (reference) had the same four movements:
    # FORWARD -> COMPARE -> BACKWARD -> UPDATE, just written for one
    # fixed hidden_size instead of a variable one.

    for epoch in range(epochs):
        total_error = 0.0

        for i in range(len(tells)):
            layer_0 = tells[i:i + 1]   # (1, n_features)
            target = strike[i:i + 1]   # (1, 1)

            # --- FORWARD ---
            layer_1 = relu(layer_0 @ weights_0_1)  # (1, hidden_size) or (1,3) but its left as hidden_size so
                                                   #                  it isnt hardcoded. I did similar things with     
            layer_2 = layer_1 @ weights_1_2        # (1, 1)           lines 30, 29, 42, 55, 58, and 59

            # --- COMPARE ---
            layer_2_error = (layer_2 - target) ** 2   # (1, 1)        
            layer_2_delta = layer_2 - target          # (1, 1)

            # --- BACKWARD ---
            layer_1_delta = (layer_2_delta @ weights_1_2.T) * relu2deriv(layer_1)  # (1, hidden_size)

            # --- UPDATE ---
            weights_1_2 -= alpha * (layer_1.T @ layer_2_delta)  # (hidden_size, 1)
            weights_0_1 -= alpha * (layer_0.T @ layer_1_delta)  # (n_features, hidden_size)

            total_error += float(np.sum(layer_2_error))

        error_history.append(total_error)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1}: total error = {total_error:.6f}")

    return weights_0_1, weights_1_2, error_history




# 2. Run with alpha=0.2, epochs=60, hidden_size=4, seed=1
print("=== Training run (alpha=0.2, epochs=60, hidden_size=4, seed=1) ===")
    weights_0_1, weights_1_2, error_history = train(
        tells, strike, alpha=0.2, epochs=60, hidden_size=4, seed=1
    )

print(f"\nFinal-epoch error: {error_history[-1]:.6f}")
print("Expected (Week 5 version): 0.000015")

# 3. Final predictions vs goals

print("\n=== Final predictions vs goals ===")
for i in range(len(tells)):
    layer_1 = relu(tells[i:i + 1] @ weights_0_1)
    layer_2 = layer_1 @ weights_1_2
    pred = layer_2.item()
    goal = strike[i].item()
    side_ok = (pred > 0.5) == (goal > 0.5)
    print(f"  Sensing {i} {tells[i]}: prediction={pred:.4f}, "
          f"goal={goal}, correct side: {side_ok}")

print("\n=== Hidden layer weights ===")
print("weights_0_1.round(2):\n", weights_0_1.round(2))
print("weights_1_2.round(2):\n", weights_1_2.round(2))


# helper func for tests - Micah
def one_step_with_shapes(layer_0, target, weights_0_1, weights_1_2):
    layer_1 = relu(layer_0 @ weights_0_1)
    layer_2 = layer_1 @ weights_1_2

    layer_2_delta = layer_2 - target
    layer_1_delta = (
        layer_2_delta @ weights_1_2.T
    ) * relu2deriv(layer_1)

    return {
        "layer_0": layer_0.shape,
        "layer_1": layer_1.shape,
        "layer_2": layer_2.shape,
        "layer_1_delta": layer_1_delta.shape,
        "layer_2_delta": layer_2_delta.shape,
        "weights_0_1": weights_0_1.shape,
        "weights_1_2": weights_1_2.shape,
    }



## RESULTS! ##
#-------------------
# week5 results: 
# Epoch 1: total error = 1.414206
# Epoch 10: total error = 0.634231
# Epoch 20: total error = 0.358384
# Epoch 30: total error = 0.083018
# Epoch 40: total error = 0.006467
# Epoch 50: total error = 0.000329
# Epoch 60: total error = 0.000015 <-- what we want to match up
# week6 results:
# === Training run (alpha=0.2, epochs=60, hidden_size=4, seed=1) ===
# Epoch 1: total error = 1.414206
# Epoch 10: total error = 0.634231
# Epoch 20: total error = 0.358384
# Epoch 30: total error = 0.083018
# Epoch 40: total error = 0.006467
# Epoch 50: total error = 0.000329
# Epoch 60: total error = 0.000015 <-- here we see that the refacted loop is
#                                      pretty much identical the origanl week5 training loop

# Part 2
The Wider Network

## Diagram 1: Wider Network

```text
layer_0 --[weights_0_1 (3, 32), relu]--> layer_1 --[weights_1_2 (32, 1)]--> layer_2
(1, 3)                                   (1, 32)                            (1, 1)
```

**Total Weights:**  
$3 \times 32 + 32 \times 1 = 96 + 32 = 128$ total weights.

A wider hidden layer allows the network to learn many diverse feature combinations simultaneously, which can speed up training on complex pattern matching tasks. However, a tiny dataset having 128 weights like the Trial dataset (that has 4 samples) carries a high risk of overfitting or memorizing noise rather than generalizing well.


## Diagram 2: Deeper Network

```text
layer_0 --[weights_0_1 (3, 4), relu]--> layer_1 --[weights_1_2 (4, 4), relu]--> layer_2 --[weights_2_3 (4, 4), relu]--> layer_3 --[weights_3_4 (4, 1)]--> layer_4
(1, 3)                                  (1, 4)                                  (1, 4)                                     (1, 4)                            (1, 1)
```
**Total Weights:**  
 $3 \times 4 + 4 \times 4 + 4 \times 4 + 4 \times 1 = 12 + 16 + 16 + 4 = 48$

 A deeper network allows each next hidden layer to combine what was learned from the previous layer to model highly complex non-linear functions. However, deeper networks are much more sensitive to random weight initialization that could cause dead ReLU units or vanishing gradients.

## Diagram 3: Multi-output Network

```text
layer_0 --[weights_0_1 (3, 8), relu]--> layer_1 --[weights_1_2 (8, 4)]--> layer_2
(1, 3)                                  (1, 8)                             (1, 4)
```
**Total Weights:**  
$3 \times 8 + 8 \times 4 = 24 + 32 = 56$

Multi-output networks can perform multi label classification that leverages shared representations learned in the hidden layer across all outputs. However if the output tasks contradict each other, gradient updates from one output target might interfere with or degrade performance on another.


## Diagram 4: Mystery Network

Ten environmental features that are processed through three hidden layers to classify the target into one of four behavior categories.

```text
layer_0 --[weights_0_1 (10, 32), relu]--> layer_1 --[weights_1_2 (32, 32), relu]--> layer_2 --[weights_2_3 (32, 16), relu]--> layer_3 --[weights_3_4 (16, 4)]--> layer_4
(1, 10)                                  (1, 32)                                  (1, 32)                                  (1, 16)                             (1, 4)
```


**Total Weights:**  
$10 \times 32 + 32 \times 32 + 32 \times 16 + 16 \times 4 = 320 + 1024 + 512 + 64 = 1,920$

The architecture is wider at first but then narrows drastically allowing high dimensional feature extraction across 10 input signals before being compressed into 4 output classes. However, the almost 2 thousand weights across the four layer would require a larger dataset to avoid overfitting.



# Part 3
Build From Diagram

In [ ]:
import numpy as np

# ==============================================================================
# 4. Comment block explaining shape constraints for each weight matrix:
# ------------------------------------------------------------------------------
# weights_0_1 shape (3, 8): Connects layer_0 (1, 3) to layer_1 (1, 8). 
#   The input dimension must match layer_0's feature count (3) to perform 
#   matrix multiplication, while the output dimension sets layer_1's width (8).
#
# weights_1_2 shape (8, 4): Connects layer_1 (1, 8) to layer_2 (1, 4).
#   The input dimension must match layer_1's hidden size (8), and the output 
#   dimension sets layer_2's width (4).
#
# weights_2_3 shape (4, 1): Connects layer_2 (1, 4) to layer_3 (1, 1).
#   The input dimension must match layer_2's hidden size (4), and the output 
#   dimension produces the scalar output of layer_3 (1).
# ==============================================================================


# 1. Implement train_from_diagram based on the architecture diagram
def train_from_diagram(tells, strike, alpha, epochs, seed):
    """
    Trains a 3-layer deep neural network (layer_0 -> layer_1 -> layer_2 -> layer_3)
    using stochastic gradient descent based strictly on the provided diagram.
    """
    np.random.seed(seed)
    
    # Initialize three weight matrices with prescribed shapes: 2 * random - 1
    weights_0_1 = 2 * np.random.random((3, 8)) - 1  # (3, 8)
    weights_1_2 = 2 * np.random.random((8, 4)) - 1  # (8, 4)
    weights_2_3 = 2 * np.random.random((4, 1)) - 1  # (4, 1)

    error_history = []

    for epoch in range(epochs):
        total_error = 0.0

        # Four sensings (samples) per epoch
        for i in range(len(tells)):
            # --- FORWARD
            layer_0 = tells[i:i+1]                      # (1, 3)
            
            layer_1_input = layer_0 @ weights_0_1        # (1, 8)
            layer_1 = np.maximum(0, layer_1_input)      # ReLU activation -> (1, 8)
            
            layer_2_input = layer_1 @ weights_1_2        # (1, 4)
            layer_2 = np.maximum(0, layer_2_input)      # ReLU activation -> (1, 4)
            
            layer_3 = layer_2 @ weights_2_3              # No activation -> (1, 1)

            # --- COMPARE
            goal = strike[i:i+1]                         # (1, 1)
            error = (layer_3 - goal) ** 2               # (1, 1)
            total_error += float(np.sum(error))

            # --- BACKWARD
            layer_3_delta = layer_3 - goal               # (1, 1)
            
            layer_2_delta = (layer_3_delta @ weights_2_3.T) * (layer_2_input > 0)  # (1, 4)
            layer_1_delta = (layer_2_delta @ weights_1_2.T) * (layer_1_input > 0)  # (1, 8)

            # --- UPDATE
            weights_2_3 -= alpha * (layer_2.T @ layer_3_delta)  # (4, 1)
            weights_1_2 -= alpha * (layer_1.T @ layer_2_delta)  # (8, 4)
            weights_0_1 -= alpha * (layer_0.T @ layer_1_delta)  # (3, 8)

        error_history.append(total_error)

    return error_history, weights_0_1, weights_1_2, weights_2_3


    # Trial dataset from Week 5
tells = np.array([
        [1, 0, 1],
        [0, 1, 1],
        [0, 0, 1],
        [1, 1, 1]
])
    
strike = np.array([
        [1],
        [1],
        [0],
        [0]
])

    # Configuration for Part 3
alpha = 0.1
epochs = 150
seed = 4

# 2. Run train_from_diagram and print total error every 30 epochs
print("--- Training Deep Network (Seed 4) ---")
np.random.seed(seed)
    
# We execute the training routine to get error history and final weights
error_history, w01, w12, w23 = train_from_diagram(tells, strike, alpha, epochs, seed)

for ep in range(epochs):
    if (ep + 1) % 30 == 0 or ep == 0:
        print(f"Epoch {ep + 1:3d} | Total Error: {error_history[ep]:.6f}")

# 3. Compute and print the four final predictions vs goals
print("\n--- Final Predictions vs Goals ---")
for i in range(len(tells)):
    layer_0 = tells[i:i+1]
    layer_1 = np.maximum(0, layer_0 @ w01)
    layer_2 = np.maximum(0, layer_1 @ w12)
    prediction = layer_2 @ w23
        
    print(f"Input: {tells[i]} | Prediction: {prediction[0, 0]:.4f} | Goal: {strike[i, 0]}")

# Holocron -- Reading

Why might the diagram discipline — the cheapest skill you’ll learn this semester
— be the most important? Refer to your own experience of Part 3: what was it like to write a
network you had never seen before, working only from the picture? Speculate (briefly) about what
Korrith’s disciples lost, in the bible’s terms

Writing a network that I have never seen before is similar to writing one that you have seen before. There is just one difference. You have to figure out what the right weights, neuron layers, input and output shapes are instead of them those numbers and shapes be given to you. But why is it the most important? It is the most important because without this talent, one would be unable to create a neural network by themselves. If one is able to identify what neural network shapes you need to set up the neural networks yourselves, then you don't need few other talents to be able to create a another neural network. If you know what size and shapes to make the neural network from a graph or picture, then you are just copying one's prior steps of creating neural networks. Same steps, new numbers.